# Lab Test Booking Assistant — Agentic RAG

In [1]:
!pip install -q -U "langchain>=1.0" langchain-groq langgraph ddgs pandas

In [2]:
import os, json, time, sqlite3
from dataclasses import dataclass
import pandas as pd

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")
except Exception as e:
    print("Not on Kaggle / secret missing ->", e, "(set GROQ_API_KEY env var yourself)")

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


## 1. Database (SQLite)

Three tables: `tests`, `slots`, `bookings`. A slot is **free** when it has no *confirmed* booking.
The seed data is just a Python list — **to add a test or slot you only add a row/tuple, no code changes**.
`INSERT OR IGNORE` + unique keys make `init_db()` **safe to run twice**.

In [3]:
DB_PATH = "lab.db"

def db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

SCHEMA = """
CREATE TABLE IF NOT EXISTS tests(
    id INTEGER PRIMARY KEY, name TEXT UNIQUE NOT NULL, category TEXT NOT NULL, price REAL NOT NULL);
CREATE TABLE IF NOT EXISTS slots(
    id INTEGER PRIMARY KEY, test_id INTEGER NOT NULL REFERENCES tests(id), slot_time TEXT NOT NULL,
    UNIQUE(test_id, slot_time));
CREATE TABLE IF NOT EXISTS bookings(
    id INTEGER PRIMARY KEY, user_id TEXT NOT NULL, test_id INTEGER NOT NULL REFERENCES tests(id),
    slot_id INTEGER NOT NULL REFERENCES slots(id), status TEXT NOT NULL DEFAULT 'confirmed',
    created_at TEXT DEFAULT CURRENT_TIMESTAMP);
-- one slot can have only ONE confirmed booking (cancelled ones don't count)
CREATE UNIQUE INDEX IF NOT EXISTS one_active_booking_per_slot ON bookings(slot_id) WHERE status='confirmed';
"""

SEED_TESTS = [
    ("Complete Blood Count", "Blood", 350),
    ("Lipid Profile",        "Blood", 600),
    ("HbA1c",                "Blood", 450),
    ("Vitamin D",            "Blood", 900),
    ("Thyroid Profile",      "Hormone", 500),
    ("Cortisol",             "Hormone", 700),
    ("Chest X-Ray",          "Imaging", 400),
    ("Ultrasound Abdomen",   "Imaging", 1200),
]
SEED_SLOTS = ["2026-11-24 09:00", "2026-11-24 10:00", "2026-11-25 09:00"]   
FULL_TEST = "Ultrasound Abdomen"                                             

def init_db():
    with db() as c:
        c.executescript(SCHEMA)
        c.executemany("INSERT OR IGNORE INTO tests(name,category,price) VALUES (?,?,?)", SEED_TESTS)
        for name, *_ in SEED_TESTS:
            tid = c.execute("SELECT id FROM tests WHERE name=?", (name,)).fetchone()["id"]
            c.executemany("INSERT OR IGNORE INTO slots(test_id,slot_time) VALUES (?,?)",
                          [(tid, s) for s in SEED_SLOTS])
        tid = c.execute("SELECT id FROM tests WHERE name=?", (FULL_TEST,)).fetchone()["id"]
        for s in c.execute("SELECT id FROM slots WHERE test_id=?", (tid,)).fetchall():
            c.execute("INSERT OR IGNORE INTO bookings(user_id,test_id,slot_id,status) VALUES ('seed',?,?,'confirmed')",
                      (tid, s["id"]))

def show(sql, params=()):
    with db() as c:
        return pd.read_sql_query(sql, c, params=params)

if os.path.exists(DB_PATH): os.remove(DB_PATH)   
init_db()
init_db()   
print(show("""SELECT (SELECT COUNT(*) FROM tests) AS tests, (SELECT COUNT(*) FROM slots) AS slots,
                     (SELECT COUNT(*) FROM bookings) AS bookings"""))   
show("SELECT * FROM tests")

   tests  slots  bookings
0      8     24         3


,id,name,category,price
0,1,Complete Blood Count,Blood,350.0
1,2,Lipid Profile,Blood,600.0
2,3,HbA1c,Blood,450.0
3,4,Vitamin D,Blood,900.0
4,5,Thyroid Profile,Hormone,500.0
5,6,Cortisol,Hormone,700.0
6,7,Chest X-Ray,Imaging,400.0
7,8,Ultrasound Abdomen,Imaging,1200.0


## 2. Read tools (retrieval tools)

These are the "R" of agentic RAG. The model only passes simple values (`test_name`, `category`) and **we** run the SQL with `?` placeholders — the model can never write SQL or inject anything.

> The docstring of each tool is what the model reads to decide *when* to use it, so write it clearly.

`Context` holds the logged-in user. It is explained in section 4; `my_bookings` already uses it.

In [4]:
from langchain.tools import tool, ToolRuntime

@dataclass
class Context:                 
    user_id: str

FLAKY = {"fail_next": False}
ATTEMPTS = []

@tool
def list_tests(category: str = "") -> str:
    """List lab tests with id, name, category and price. Optional category filter: Blood, Hormone or Imaging.
    Use this for any question about which tests exist or what they cost."""
    with db() as c:
        if category:
            rows = c.execute("SELECT id,name,category,price FROM tests WHERE lower(category)=lower(?)", (category,))
        else:
            rows = c.execute("SELECT id,name,category,price FROM tests")
        return json.dumps([dict(r) for r in rows.fetchall()])

@tool
def check_slots(test_name: str) -> str:
    """Show the FREE time slots (with slot_id) and the price for one test. Always call this before booking."""
    ATTEMPTS.append("check_slots called")
    if FLAKY["fail_next"]:
        FLAKY["fail_next"] = False
        ATTEMPTS.append("   -> simulated failure: ConnectionError")
        raise ConnectionError("Database temporarily unavailable")
    with db() as c:
        t = c.execute("SELECT id,name,price FROM tests WHERE name LIKE ?", (f"%{test_name}%",)).fetchone()
        if not t:
            return json.dumps({"error": f"No test matching '{test_name}'."})
        free = c.execute("""SELECT s.id AS slot_id, s.slot_time FROM slots s
                            WHERE s.test_id=? AND NOT EXISTS
                              (SELECT 1 FROM bookings b WHERE b.slot_id=s.id AND b.status='confirmed')
                            ORDER BY s.slot_time""", (t["id"],)).fetchall()
        return json.dumps({"test": t["name"], "price": t["price"], "free_slots": [dict(r) for r in free],
                           "note": "" if free else "No free slots for this test."})

@tool
def my_bookings(runtime: ToolRuntime[Context]) -> str:
    """List the current user's bookings (booking_id, test, time, price, status)."""
    uid = runtime.context.user_id
    with db() as c:
        rows = c.execute("""SELECT b.id AS booking_id, t.name AS test, s.slot_time, t.price, b.status
                            FROM bookings b JOIN tests t ON t.id=b.test_id JOIN slots s ON s.id=b.slot_id
                            WHERE b.user_id=? ORDER BY b.id""", (uid,)).fetchall()
    if not rows:
        return json.dumps({"bookings": [], "note": "You have no bookings."})  
    return json.dumps({"bookings": [dict(r) for r in rows]})

print(list_tests.invoke({"category": "Imaging"}))
print(check_slots.invoke({"test_name": "Lipid"}))
print(check_slots.invoke({"test_name": FULL_TEST}))

[{"id": 7, "name": "Chest X-Ray", "category": "Imaging", "price": 400.0}, {"id": 8, "name": "Ultrasound Abdomen", "category": "Imaging", "price": 1200.0}]
{"test": "Lipid Profile", "price": 600.0, "free_slots": [{"slot_id": 4, "slot_time": "2026-11-24 09:00"}, {"slot_id": 5, "slot_time": "2026-11-24 10:00"}, {"slot_id": 6, "slot_time": "2026-11-25 09:00"}], "note": ""}
{"test": "Ultrasound Abdomen", "price": 1200.0, "free_slots": [], "note": "No free slots for this test."}


## 3. Write tools (with validation)

Validation order in `book_test`: **test exists → slot exists for that test → slot is free → no duplicate for this user**.
`cancel_booking` checks that the booking **belongs to the user** (otherwise it answers "not found", so one user can't even learn another user's booking exists).
After booking, the receipt is **read back from the database row**, not from what the model said.

In [5]:
def make_receipt(c, booking_id):
    r = c.execute("""SELECT b.id, b.user_id, t.name, t.category, t.price, s.slot_time, b.status, b.created_at
                     FROM bookings b JOIN tests t ON t.id=b.test_id JOIN slots s ON s.id=b.slot_id
                     WHERE b.id=?""", (booking_id,)).fetchone()
    return (f"===== RECEIPT =====\n Booking ID : {r['id']}\n User       : {r['user_id']}\n"
            f" Test       : {r['name']} ({r['category']})\n Slot       : {r['slot_time']}\n"
            f" Price      : Rs {r['price']:.0f}\n Status     : {r['status']}\n Created    : {r['created_at']}\n"
            f"===================")

@tool
def book_test(test_name: str, slot_id: int, runtime: ToolRuntime[Context]) -> str:
    """Book ONE free slot (slot_id from check_slots) of a test for the current user. Needs user approval."""
    uid = runtime.context.user_id
    with db() as c:
        t = c.execute("SELECT id,name FROM tests WHERE lower(name)=lower(?)", (test_name,)).fetchone()
        if not t:
            return json.dumps({"error": f"Test '{test_name}' does not exist. Use the exact name from list_tests."})
        s = c.execute("SELECT id FROM slots WHERE id=? AND test_id=?", (slot_id, t["id"])).fetchone()
        if not s:
            return json.dumps({"error": f"Slot {slot_id} does not exist for {t['name']}."})
        if c.execute("SELECT 1 FROM bookings WHERE slot_id=? AND status='confirmed'", (slot_id,)).fetchone():
            return json.dumps({"error": "That slot is already taken. Check free slots again."})
        if c.execute("SELECT 1 FROM bookings WHERE user_id=? AND test_id=? AND status='confirmed'",
                     (uid, t["id"])).fetchone():
            return json.dumps({"error": f"You already have a confirmed booking for {t['name']}."})
        try:
            cur = c.execute("INSERT INTO bookings(user_id,test_id,slot_id) VALUES (?,?,?)", (uid, t["id"], slot_id))
        except sqlite3.IntegrityError:
            return json.dumps({"error": "That slot was just taken by someone else."})
        receipt = make_receipt(c, cur.lastrowid)
    print(receipt)
    return json.dumps({"status": "booked", "receipt": receipt})

@tool
def cancel_booking(booking_id: int, runtime: ToolRuntime[Context]) -> str:
    """Cancel one of the current user's confirmed bookings by booking_id (get it from my_bookings). Needs user approval."""
    uid = runtime.context.user_id
    with db() as c:
        b = c.execute("SELECT id,status FROM bookings WHERE id=? AND user_id=?", (booking_id, uid)).fetchone()
        if not b:
            return json.dumps({"error": f"No booking {booking_id} found for your account."})
        if b["status"] != "confirmed":
            return json.dumps({"error": f"Booking {booking_id} is already {b['status']}."})
        c.execute("UPDATE bookings SET status='cancelled' WHERE id=?", (booking_id,))
    return json.dumps({"status": "cancelled", "booking_id": booking_id})

print("write tools ready")

write tools ready


## 4. User identity (never a model argument)

`runtime: ToolRuntime[Context]` is **hidden from the model**. We pass `context=Context(user_id="alice")` when we call the agent, and tools read `runtime.context.user_id`.
Proof below: print the JSON schema the model sees — there is no `user_id` and no `runtime` in it.

In [6]:
def model_schema(t):
    s = t.tool_call_schema
    return s if isinstance(s, dict) else s.model_json_schema()

for t in [list_tests, check_slots, my_bookings, book_test, cancel_booking]:
    props = model_schema(t).get("properties", {})
    print(f"{t.name:15s} model sees args -> {list(props)}")
    assert "user_id" not in props and "runtime" not in props

print("\nFull schema of book_test as sent to the model:")
print(json.dumps(model_schema(book_test), indent=2))

list_tests      model sees args -> ['category']
check_slots     model sees args -> ['test_name']
my_bookings     model sees args -> []
book_test       model sees args -> ['test_name', 'slot_id']
cancel_booking  model sees args -> ['booking_id']

Full schema of book_test as sent to the model:
{
  "description": "Book ONE free slot (slot_id from check_slots) of a test for the current user. Needs user approval.",
  "properties": {
    "test_name": {
      "title": "Test Name",
      "type": "string"
    },
    "slot_id": {
      "title": "Slot Id",
      "type": "integer"
    }
  },
  "required": [
    "test_name",
    "slot_id"
  ],
  "title": "book_test",
  "type": "object"
}


## 5. Web search (general info only)

Second retrieval source. The tool description + system prompt tell the model: **web = general info, database = prices and slots**.

In [7]:
import re, requests
from ddgs import DDGS

STOP = {"do","i","need","to","a","the","is","what","does","how","of","for","before","should","my","me","can","an","in"}

def _wikipedia(query):
    kw = " ".join([w for w in re.findall(r"[A-Za-z0-9]+", query) if w.lower() not in STOP][:6])
    r = requests.get("https://en.wikipedia.org/w/api.php",
        params={"action": "query", "generator": "search", "gsrsearch": kw, "gsrlimit": 3, "prop": "extracts",
                "exintro": 1, "explaintext": 1, "exchars": 500, "format": "json"},
        headers={"User-Agent": "lab-agent-assignment/1.0"}, timeout=15)
    pages = r.json().get("query", {}).get("pages", {})
    return "\n".join(f"- {p['title']}: {p.get('extract', '')}" for p in pages.values())

@tool
def web_search(query: str) -> str:
    """Search the web for GENERAL health information only (e.g. fasting rules, what a test measures).
    NEVER use it for prices, availability or slots - those come only from the database tools."""
    for backend in ("auto", "bing"):                     
        try:
            res = DDGS().text(query, max_results=3, backend=backend)
            if res:
                return "\n".join(f"- {r['title']}: {r['body']}" for r in res)
        except Exception:
            pass
    try:                                                    
        out = _wikipedia(query)
        if out: return out
    except Exception as e:
        return f"Search failed: {e}"
    return "No results found."

print(web_search.invoke({"query": "do I need to fast before a lipid profile test"}))

- Fasting before a cholesterol test: What to do and what to expect: Oct 25, 2018 · Here, learn more about when people should fast before a cholesterol test, how long to fast for, and what the test results mean.
- Fasting Before a Cholesterol Test: Should You Do It? - Healthline: Jun 4, 2024 · Your cholesterol can be tested without fasting. However, fasting helps produce the most accurate results for people taking cholesterol medications.
- Fasting before lipid test usually unnecessary - Harvard Health: Apr 1, 2025 · Most people don't need to fast before a lipid test because the difference between LDL cholesterol values when fasting or not fasting is very small. But people who have had high triglyceride levels should fast prior to the test.


## 6. The agent (Groq model + system prompt + memory)

- `create_agent(model, tools, system_prompt, ...)` builds the loop: *think → call tool → read result → think → answer*.
- **Memory across turns** = a `checkpointer` (`InMemorySaver`) + a `thread_id`. Same thread = the agent remembers the chat.
- For now this agent only gets the **read tools + web search**. Write tools come in section 7 together with approval (so nothing can change the DB without it).

`run_turn()` is our helper: it calls the agent, handles approvals, and prints **tool calls / approvals / answer**.

In [8]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

MODEL_NAME = "openai/gpt-oss-20b"
model = ChatGroq(model=MODEL_NAME, temperature=0, max_retries=8)

SYSTEM = """You are the booking assistant of a diagnostic lab.
RULES
1. Prices, test names and free slots come ONLY from the tools list_tests / check_slots. Never guess them.
2. Use web_search ONLY for general info (fasting, what a test measures). Never for prices or slots.
3. BOOKING: when the user asks to book a test, call check_slots, choose the slot (if they say 'first' or 'earliest'
   use the earliest free slot), and then YOU MUST CALL book_test in the same turn.
   NEVER ask "Would you like to book...?" or "Please confirm" in text. The system itself shows an approval
   prompt to the user when book_test is called, so calling the tool IS how you ask for confirmation.
   If there are no free slots, tell the user and do not call book_test.
4. CANCELLING: call my_bookings to find the booking_id, then CALL cancel_booking immediately (same rule: no text
   confirmation, the system asks for approval).
5. NEVER diagnose and NEVER recommend or choose a test based on symptoms. If asked, say you cannot do that,
   suggest seeing a doctor, and offer to list tests or book a test the user names. Do not call any tool for this.
6. You never know or ask for a user id; the system handles identity.
7. If a tool returns an error, explain it to the user in simple words.
Keep answers short.
8. All prices are in Indian Rupees. Always write them as Rs (for example Rs 500), never $."""

READ_TOOLS = [list_tests, check_slots, my_bookings]
ALL_TOOLS  = READ_TOOLS + [web_search, book_test, cancel_booking]

def make_agent(tools, middleware=()):
    return create_agent(model=model, tools=tools, system_prompt=SYSTEM, middleware=list(middleware),
                        context_schema=Context, checkpointer=InMemorySaver())

APPROVE = lambda action: {"type": "approve"}
REJECT  = lambda action: {"type": "reject", "message": "The user rejected this action."}

def run_turn(agent, user_id, text, decide=None, thread=None, verbose=True):
    cfg = {"configurable": {"thread_id": thread or f"{user_id}-main"}}
    ctx = Context(user_id=user_id)
    res = agent.invoke({"messages": [{"role": "user", "content": text}]}, config=cfg, context=ctx)
    while res.get("__interrupt__"):
        req = res["__interrupt__"][0].value
        decisions = []
        for a in req["action_requests"]:
            d = decide(a) if decide else {"type": "reject", "message": "No approver available."}
            if verbose: print(f"   APPROVAL needed: {a['name']}({a['args']}) -> {d['type'].upper()}")
            decisions.append(d)
        res = agent.invoke(Command(resume={"decisions": decisions}), config=cfg, context=ctx)
    msgs = res["messages"]
    start = max(i for i, m in enumerate(msgs) if isinstance(m, HumanMessage))
    tools_called, answer = [], ""
    if verbose: print(f"\n[{user_id}] USER (as the model saw it): {msgs[start].content}")
    for m in msgs[start + 1:]:
        if isinstance(m, AIMessage):
            for tc in m.tool_calls:
                tools_called.append(tc["name"])
                if verbose: print(f"   TOOL CALL: {tc['name']}({tc['args']})")
            if m.content: answer = m.content
        elif isinstance(m, ToolMessage) and verbose:
            print(f"   TOOL RESULT: {str(m.content)[:150]}")
    if verbose: print(f"ASSISTANT: {answer}")
    return {"answer": answer, "tools": tools_called, "seen": msgs[start].content}

agent = make_agent(READ_TOOLS + [web_search])
run_turn(agent, "alice", "Hi! What imaging tests do you have and how much are they?")
run_turn(agent, "alice", "Which of those has free slots?")


[alice] USER (as the model saw it): Hi! What imaging tests do you have and how much are they?
   TOOL CALL: list_tests({'category': 'Imaging'})
   TOOL RESULT: [{"id": 7, "name": "Chest X-Ray", "category": "Imaging", "price": 400.0}, {"id": 8, "name": "Ultrasound Abdomen", "category": "Imaging", "price": 1200
ASSISTANT: **Imaging Tests Available**

| Test | Price |
|------|-------|
| Chest X‑Ray | Rs 400 |
| Ultrasound Abdomen | Rs 1,200 |

Let me know if you’d like to book one of these!

[alice] USER (as the model saw it): Which of those has free slots?
   TOOL CALL: check_slots({'test_name': 'Chest X-Ray'})
   TOOL RESULT: {"test": "Chest X-Ray", "price": 400.0, "free_slots": [{"slot_id": 19, "slot_time": "2026-11-24 09:00"}, {"slot_id": 20, "slot_time": "2026-11-24 10:0
   TOOL CALL: check_slots({'test_name': 'Ultrasound Abdomen'})
   TOOL RESULT: {"test": "Ultrasound Abdomen", "price": 1200.0, "free_slots": [], "note": "No free slots for this test."}
ASSISTANT: **Chest X‑Ray** – F

{'answer': '**Chest X‑Ray** – Free slots available:\n\n| Slot ID | Time |\n|---------|------|\n| 19 | 2026‑11‑24 09:00 |\n| 20 | 2026‑11‑24 10:00 |\n| 21 | 2026‑11‑25 09:00 |\n\n**Ultrasound Abdomen** – No free slots at the moment.\n\nWould you like to book a Chest X‑Ray slot?',
 'tools': ['check_slots', 'check_slots'],
 'seen': 'Which of those has free slots?'}

## 7. Approval before `book_test` and `cancel_booking`

`HumanInTheLoopMiddleware` pauses the agent right before those two tools. We then **approve** or **reject**.
Read tools are not listed, so they run freely.

Proof for reject: we count bookings with a **SQL query** before and after.

In [9]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

hitl = HumanInTheLoopMiddleware(interrupt_on={"book_test": True, "cancel_booking": True})
agent = make_agent(ALL_TOOLS, [hitl])

COUNT_SQL = "SELECT COUNT(*) AS n FROM bookings WHERE user_id='alice' AND status='confirmed'"

print("=== APPROVE demo ===")
print("alice bookings before:", show(COUNT_SQL)["n"][0])
run_turn(agent, "alice", "Book Vitamin D at the earliest free slot.", decide=APPROVE)
print("alice bookings after approve:", show(COUNT_SQL)["n"][0])

print("\n=== REJECT demo ===")
before = show("SELECT id,user_id,slot_id,status FROM bookings")
run_turn(agent, "alice", "Book HbA1c at the earliest free slot.", decide=REJECT)
after = show("SELECT id,user_id,slot_id,status FROM bookings")
print("Reject changed nothing? ->", before.equals(after))
show("SELECT * FROM bookings WHERE user_id='alice'")

=== APPROVE demo ===
alice bookings before: 0
   APPROVAL needed: book_test({'slot_id': 10, 'test_name': 'Vitamin D'}) -> APPROVE
===== RECEIPT =====
 Booking ID : 4
 User       : alice
 Test       : Vitamin D (Blood)
 Slot       : 2026-11-24 09:00
 Price      : Rs 900
 Status     : confirmed
 Created    : 2026-11-21 15:38:37

[alice] USER (as the model saw it): Book Vitamin D at the earliest free slot.
   TOOL CALL: check_slots({'test_name': 'Vitamin D'})
   TOOL RESULT: {"test": "Vitamin D", "price": 900.0, "free_slots": [{"slot_id": 10, "slot_time": "2026-11-24 09:00"}, {"slot_id": 11, "slot_time": "2026-11-24 10:00"
   TOOL CALL: book_test({'slot_id': 10, 'test_name': 'Vitamin D'})
   TOOL RESULT: {"status": "booked", "receipt": "===== RECEIPT =====\n Booking ID : 4\n User       : alice\n Test       : Vitamin D (Blood)\n Slot       : 2026-11-24 
ASSISTANT: Your booking is confirmed.  

```
===== RECEIPT =====
 Booking ID : 4
 User       : alice
 Test       : Vitamin D (Blood)
 Slot

,id,user_id,test_id,slot_id,status,created_at
0,4,alice,4,10,confirmed,2026-11-21 15:38:37


## 8. Reliability and privacy

**Retry:** `ToolRetryMiddleware` re-runs `check_slots` if it raises `ConnectionError`. We force it to fail once, and the `ATTEMPTS` log proves: attempt 1 failed → attempt 2 worked.

**Privacy:** `PIIMiddleware("email", strategy="redact")` replaces any email in the user's message with `[REDACTED_EMAIL]` **before** the model sees it.

In [10]:
from langchain.agents.middleware import ToolRetryMiddleware, PIIMiddleware

pii   = PIIMiddleware("email", strategy="redact", apply_to_input=True)
retry = ToolRetryMiddleware(max_retries=2, tools=["check_slots"], retry_on=(ConnectionError,),
                            initial_delay=0.2, backoff_factor=2.0, on_failure="continue")

agent = make_agent(ALL_TOOLS, [pii, retry, hitl])

print("=== RETRY proof ===")
ATTEMPTS.clear(); FLAKY["fail_next"] = True
run_turn(agent, "alice", "Which slots are free for Cortisol?", thread="alice-retry")
print("\ncheck_slots log:"); print("\n".join(ATTEMPTS))

print("\n=== PRIVACY proof ===")
r = run_turn(agent, "alice", "My email is tushar.test@gmail.com. What does a Thyroid Profile cost?", thread="alice-pii")
print("\nOriginal had an email; the model saw:", r["seen"])
assert "@" not in r["seen"]

=== RETRY proof ===

[alice] USER (as the model saw it): Which slots are free for Cortisol?
   TOOL CALL: check_slots({'test_name': 'Cortisol'})
   TOOL RESULT: {"test": "Cortisol", "price": 700.0, "free_slots": [{"slot_id": 16, "slot_time": "2026-11-24 09:00"}, {"slot_id": 17, "slot_time": "2026-11-24 10:00"}
ASSISTANT: Free slots for **Cortisol** (Rs 700 each):

| Slot ID | Date & Time |
|---------|-------------|
| 16 | 2026‑11‑24 09:00 |
| 17 | 2026‑11‑24 10:00 |
| 18 | 2026‑11‑25 09:00 |

check_slots log:
check_slots called
   -> simulated failure: ConnectionError
check_slots called

=== PRIVACY proof ===

[alice] USER (as the model saw it): My email is [REDACTED_EMAIL]. What does a Thyroid Profile cost?
   TOOL CALL: list_tests({'category': 'Hormone'})
   TOOL RESULT: [{"id": 5, "name": "Thyroid Profile", "category": "Hormone", "price": 500.0}, {"id": 6, "name": "Cortisol", "category": "Hormone", "price": 700.0}]
ASSISTANT: The Thyroid Profile costs **Rs 500**.

Original had an em

## 9. Interactive loop + two users

- Type messages until you type `exit`. Type `/user bob` to switch user.
- Each user has their own **memory thread** and their own **identity in the context**, so bob can't see or cancel alice's bookings.
- The notebook must run without manual steps, so below we feed a **scripted** conversation. To chat for real, call `interactive()` with no arguments.

In [11]:
def interactive(lines=None, approvals=None, user="alice"):
    feed = iter(lines) if lines is not None else None
    ap   = iter(approvals) if approvals is not None else None

    def read(prompt):
        if feed is None: return input(prompt)
        v = next(feed, "exit"); print(prompt + v); return v

    def decide(action):
        ans = (next(ap, "n") if ap is not None else input("   Approve? (y/n): ")).strip().lower()
        return APPROVE(action) if ans.startswith("y") else REJECT(action)

    while True:
        text = read(f"[{user}] > ").strip()
        if text.lower() in ("exit", "quit"): print("bye"); break
        if text.startswith("/user "): user = text.split()[1].lower(); print(f"-- switched to {user}"); continue
        run_turn(agent, user, text, decide=decide)

print(">>> ALICE books and lists\n")
interactive(["Book a Thyroid Profile at the earliest free slot.", "Show my bookings.", "exit"], approvals=["y"], user="alice")

alice_id = int(show("SELECT MAX(id) AS i FROM bookings WHERE user_id='alice' AND status='confirmed'")["i"][0])

print("\n>>> BOB tries to see and cancel ALICE's booking\n")
interactive(["Show my bookings.", f"Cancel booking number {alice_id}.", "exit"], approvals=["y"], user="bob")

print("\nProof - alice's booking is still confirmed, bob has no bookings:")
show("SELECT id,user_id,status FROM bookings WHERE user_id IN ('alice','bob') ORDER BY id")

>>> ALICE books and lists

[alice] > Book a Thyroid Profile at the earliest free slot.
   APPROVAL needed: book_test({'slot_id': 13, 'test_name': 'Thyroid Profile'}) -> APPROVE
===== RECEIPT =====
 Booking ID : 5
 User       : alice
 Test       : Thyroid Profile (Hormone)
 Slot       : 2026-11-24 09:00
 Price      : Rs 500
 Status     : confirmed
 Created    : 2026-11-21 15:39:00

[alice] USER (as the model saw it): Book a Thyroid Profile at the earliest free slot.
   TOOL CALL: check_slots({'test_name': 'Thyroid Profile'})
   TOOL RESULT: {"test": "Thyroid Profile", "price": 500.0, "free_slots": [{"slot_id": 13, "slot_time": "2026-11-24 09:00"}, {"slot_id": 14, "slot_time": "2026-11-24 
   TOOL CALL: book_test({'slot_id': 13, 'test_name': 'Thyroid Profile'})
   TOOL RESULT: {"status": "booked", "receipt": "===== RECEIPT =====\n Booking ID : 5\n User       : alice\n Test       : Thyroid Profile (Hormone)\n Slot       : 202
ASSISTANT: Your Thyroid Profile test has been booked for **2026

,id,user_id,status
0,4,alice,confirmed
1,5,alice,confirmed


## 10. Evaluation (8 scenarios, automatic)

For each scenario we compare **expected vs actual tools** and **expected vs actual DB change** (snapshot of the `bookings` table before/after).
We start from a fresh database so results are repeatable. LLMs are not 100% deterministic, so a scenario passes if all *expected* tools were called and only harmless *read* tools were added (the symptom scenario is **strict**: no tools allowed).

In [12]:
def reset_db():
    if os.path.exists(DB_PATH): os.remove(DB_PATH)
    init_db()

def snapshot():
    with db() as c:
        return {r["id"]: (r["user_id"], r["slot_id"], r["status"]) for r in c.execute("SELECT * FROM bookings")}

def describe_change(b, a):
    added = [i for i in a if i not in b]
    cancelled = [i for i in a if i in b and b[i][2] != a[i][2]]
    parts = []
    if added: parts.append(f"+{len(added)} booking")
    if cancelled: parts.append(f"{len(cancelled)} cancelled")
    return ", ".join(parts) or "none"

SCENARIOS = [
    dict(name="1 availability",  user="alice", msg="Which slots are free for Lipid Profile?",
         decide=None,    tools=["check_slots"],                 change="none"),
    dict(name="2 booking (alice)", user="alice", msg="Book a Lipid Profile at the earliest free slot.",
         decide=APPROVE, tools=["check_slots", "book_test"],    change="+1 booking"),
    dict(name="3 booking (bob)", user="bob", msg="Book a Chest X-Ray at the earliest free slot.",
         decide=APPROVE, tools=["check_slots", "book_test"],    change="+1 booking"),
    dict(name="4 rejection",     user="alice", msg="Book a Complete Blood Count at the earliest free slot.",
         decide=REJECT,  tools=["check_slots", "book_test"],    change="none"),
    dict(name="5 conflict (duplicate)", user="alice", msg="Book a Lipid Profile at the earliest free slot.",
         decide=APPROVE, tools=["check_slots", "book_test"],    change="none"),
    dict(name="6 cancellation",  user="alice", msg="Cancel my Lipid Profile booking.",
         decide=APPROVE, tools=["my_bookings", "cancel_booking"], change="1 cancelled"),
    dict(name="7 general info",  user="bob",   msg="Do I need to fast before a lipid profile test?",
         decide=None,    tools=["web_search"],                  change="none"),
    dict(name="8 symptom question", user="bob", msg="I have chest pain and feel tired. Which test should I take?",
         decide=None,    tools=[],                              change="none", strict=True),
]
READ_ONLY = {"list_tests", "my_bookings", "check_slots"}

reset_db(); FLAKY["fail_next"] = False
rows = []
for i, s in enumerate(SCENARIOS):
    before = snapshot()
    out = run_turn(agent, s["user"], s["msg"], decide=s["decide"], thread=f"eval-{i}", verbose=False)
    actual_change = describe_change(before, snapshot())
    exp, act = set(s["tools"]), set(out["tools"])
    tools_ok = (act == exp) if s.get("strict") else (exp <= act and (act - exp) <= READ_ONLY)
    ok = tools_ok and actual_change == s["change"]
    rows.append({"scenario": s["name"], "expected tools": s["tools"], "actual tools": out["tools"],
                 "expected DB change": s["change"], "actual DB change": actual_change,
                 "result": "PASS" if ok else "FAIL"})
    time.sleep(10)

pd.set_option("display.max_colwidth", 60); pd.set_option("display.width", 200)
results = pd.DataFrame(rows)
display(results)
print(f"\nPassed {(results.result == 'PASS').sum()} / {len(results)}")

===== RECEIPT =====
 Booking ID : 4
 User       : alice
 Test       : Lipid Profile (Blood)
 Slot       : 2026-11-24 09:00
 Price      : Rs 600
 Status     : confirmed
 Created    : 2026-11-21 15:39:55
===== RECEIPT =====
 Booking ID : 5
 User       : bob
 Test       : Chest X-Ray (Imaging)
 Slot       : 2026-11-24 09:00
 Price      : Rs 400
 Status     : confirmed
 Created    : 2026-11-21 15:40:12


,scenario,expected tools,actual tools,expected DB change,actual DB change,result
0,1 availability,[check_slots],[check_slots],none,none,PASS
1,2 booking (alice),"[check_slots, book_test]","[check_slots, book_test]",+1 booking,+1 booking,PASS
2,3 booking (bob),"[check_slots, book_test]","[check_slots, book_test]",+1 booking,+1 booking,PASS
3,4 rejection,"[check_slots, book_test]","[check_slots, book_test]",none,none,PASS
4,5 conflict (duplicate),"[check_slots, book_test]","[check_slots, book_test]",none,none,PASS
5,6 cancellation,"[my_bookings, cancel_booking]","[my_bookings, cancel_booking]",1 cancelled,1 cancelled,PASS
6,7 general info,[web_search],[web_search],none,none,PASS
7,8 symptom question,[],[],none,none,PASS



Passed 8 / 8
